# Prosody verifier for Shaer: main notebook

**Notebook version 1.6.0** · updated 2026-11-25 UTC. Section 2 prints the version of the code it downloaded and warns if this notebook copy is too old for it.

Research line on top of the reproduced Shaer baseline: a formal Arabic prosody verifier (meter, qafiya, radeef,
poem-level consistency) used to audit the baseline and to guide generation (idea 1: verifier-guided decoding).

* All code for this research lives in [`prosody-verifier/`](https://github.com/thebazshah/Shaer/tree/dev/prosody-verifier) on branch `dev`.
  Every cell is plain Python calling `prosody-verifier/pv_colab.py`, which reuses the baseline module `colab/shaer_colab.py`
  (unchanged) for the environment, progress bars, heartbeat and stall detection. Only the installation cell uses `!` commands.
* Everything this notebook writes goes to **`MyDrive/shaer-prosody/`**. Baseline results are only read, from
  `MyDrive/shaer-repro-yehia/runs/` where the baseline notebook put them.
* **Saving to Drive.** Every step works on the VM's own disk and uploads its results to `MyDrive/shaer-prosody/` through the
  Drive API, each file checked by MD5 (`☁ Drive: everything this step wrote is on Drive`): in the background about every minute,
  and once more when the step finishes or is stopped (a stop then halts *Run all*). A step first copies back from Drive what it
  resumes from, so a new session continues exactly where the last upload left it. The first step of a session asks once for
  Google's permission to use Drive. (The Drive mount alone uploads lazily: a re-run could resume from an older point.)
* Every long step shows progress and a heartbeat; a step that is silent and idle for `STALL_MINUTES` is stopped with a message,
  and re-running the cell resumes.

**Runtime.** M0, M1, M3 and all scoring need no GPU: pick a CPU runtime (cheapest). Generation (M4, M5) and training (M5) need a GPU; an A100 is fastest, an L4 works with smaller batches.

| Milestone | What | Runtime | Status |
|---|---|---|---|
| M0 | Headroom audit of the baseline poems and the human reference poems | CPU, minutes | in this notebook |
| M1 | The prosody verifier: rule tables, scansion, rhyme, poem verdicts, validation | CPU | in this notebook |
| M2 | Expert-checked gold set and sign-off of the rule tables | CPU | added later |
| M3 | Strict re-scoring of the baselines with the verifier | CPU, about 1 hour per system | in this notebook |
| M4 | Verifier-guided decoding (idea 1) vs best-of-N at equal compute | GPU, then CPU | in this notebook |
| M5 | Self-training (idea 2): rhyme letter in the prompt (2a), verifier-clean data (2b), rejection-sampling rounds (2c) | GPU (training, generation) and CPU | in this notebook |
| M6 | Quality study: LLM judge, copying and diversity, blinded human ratings | CPU (the judge is a paid API) | in this notebook |

## 1. Configuration, Google Drive, secrets

In [1]:
import getpass
from google.colab import drive, userdata

NOTEBOOK_VERSION = "1.6.0"   # updated 2026-11-25 UTC; section 2 checks it against the downloaded code

# ---- edit these -------------------------------------------------------------------------------------------
REPO_URL = "https://github.com/thebazshah/Shaer.git"
BRANCH = "dev"
BASE_MODEL = "Navid-AI/Yehia-7B-preview"   # the paper's model; it selects the baseline folders (shaer-repro-yehia)
STALL_MINUTES = 15     # a step that is silent AND idle for this long is stopped, with instructions to resume
# -----------------------------------------------------------------------------------------------------------

drive.mount("/content/drive")

def secret(name, prompt=None):
    try:
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass.getpass(prompt) if prompt else ""

HF_TOKEN = secret("HF_TOKEN")            # optional for M0 (the released poems and the meter classifier are public)
GITHUB_TOKEN = secret("GITHUB_TOKEN")    # only needed if the fork is private
OPENROUTER_API_KEY = secret("OPENROUTER_API_KEY")   # only for the LLM judge (later milestones)
print("HF token set:", bool(HF_TOKEN), "| OpenRouter key set:", bool(OPENROUTER_API_KEY))

Mounted at /content/drive
HF token set: True | OpenRouter key set: False


## 2. Get the code from GitHub (branch `dev`)

In [2]:
import importlib, pathlib, subprocess, sys

REPO = pathlib.Path("/content/Shaer")
git_url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only", git_url, BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, git_url, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)   # keep no token in .git/config

for p in (REPO / "colab", REPO / "prosody-verifier"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
import shaer_colab as sc
import pv_colab as pv
importlib.reload(sc); importlib.reload(pv)      # pick up the latest versions after a pull

cfg = sc.make_config(base_model=BASE_MODEL, stall_minutes=STALL_MINUTES, repo_url=REPO_URL, branch=BRANCH,
                     hf_token=HF_TOKEN, openrouter_key=OPENROUTER_API_KEY)
# cfg is the baseline's configuration: its folders (MyDrive/shaer-repro-yehia: the E3 adapter in sft_outputs/train/<RUN_NAME>,
# the generations in runs/e0..e3) are only read here. pv.make_config replaces it by a copy whose outputs all go to the sibling
# MyDrive/shaer-repro-yehia-prosody (run names prosody_*) and points the repository's sft/outputs and runs/ links there.
pvc = pv.make_config(cfg)            # this research writes to MyDrive/shaer-prosody (and the sibling above for training)
cfg = pvc.sc_cfg
pv.check_version(NOTEBOOK_VERSION)  # warns if this notebook copy is too old for the downloaded code
pv.runtime_info()

PAPER FLOW: Yehia-7B-preview. E0, E1, E2, E3 run as in the paper.
base model: Navid-AI/Yehia-7B-preview | Drive folder: /content/drive/MyDrive/shaer-repro-yehia | a silent, idle step is stopped after 15 min
code: v1.16.3 (updated 2026-11-25 UTC) | branch dev | commit 393e207 2026-11-27 07:12 UTC Training ETA counts evaluation and checkpoint pauses after 5 minutes (baseline code 1.16.3) | outputs -> /content/drive/MyDrive/shaer-repro-yehia-prosody
prosody-verifier code v1.5.4 | outputs -> /content/pv_local/shaer-prosody
baseline folder (read only) -> /content/drive/MyDrive/shaer-repro-yehia; anything trained or generated with the baseline's tools -> /content/drive/MyDrive/shaer-repro-yehia-prosody
✔ notebook v1.5.4 works with the prosody code v1.5.4 (updated 2026-11-25 UTC)
runtime: NVIDIA L4, 23034 MiB
  M0 needs no GPU: a CPU runtime is enough and costs the fewest compute units.
  Idea 1's generation runs (later) need a GPU: an L4 (24 GB) works with smaller batches; an A100 is faster 

## 3. Environment (about 5 minutes on a fresh runtime)

The same isolated environment as the baseline notebook (`/content/venv`, the repository's pinned versions). Safe to run again:
when everything is installed it finishes in seconds.

In [3]:
!pip -q install uv
!uv venv --python 3.12 --allow-existing /content/venv
!uv pip install --python /content/venv/bin/python torch==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/venv/bin/python -r /content/Shaer/colab/requirements-colab.txt
!uv pip install --python /content/venv/bin/python pytest
!uv pip check --python /content/venv/bin/python

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: venv
Activate with: source venv/bin/activate
Using Python 3.12.3 environment at: venv
Resolved 21 packages in 1.64s
Prepared 21 packages in 20.97s
Installed 21 packages in 179ms
 + filelock==4.2.4
 + fsspec==2026.9.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.7
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==8.9.2.26
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.20.5
 + nvidia-nvjitlink-cu12==12.9.86
 + nvidia-nvtx-cu12==12.1.105
 + sympy==1.14.0
 + torch==2.3.1+cu121
 + typing-extensions==4.16.0
Using Python 3.12.3 environment at: venv
Resolved 79 packages in 441ms
Prepared 59 packages in 3.25s
Uninstalled 1 package in 1ms
Installed 59 package

In [4]:
RUN_MO = False
RUN_M1 = False
RUN_M2 = False
RUN_M3 = True
RUN_M4 = True
RUN_M5 = True
RUN_M6 = False
RELEASE_RUNTIME = False   # True: the last cell ends the session (stops the compute units) once everything is on Drive


## 4. M0 - headroom audit

Where do the baseline poems fail, and how far are they from the human poems on the same prompts?

* **Systems:** the human reference poems of the 3,481 test prompts, the released Shaer poems (E0), and every complete baseline
  run found on Drive (E1 released adapter, E2 untuned base model, ...). Runs with several samples per prompt are audited on the
  first sample (the paper's protocol).
* **Count:** exact / too short / too long / odd final hemistich, by requested length.
* **Meter:** the repository's BiLSTM classifier per bayt: share of bayts on the requested meter, poems with every bayt on it,
  poems whose bayts get more than one meter, error rate by bayt position and by meter. Scored with a batched wrapper of the
  repository's own scorer (identical results, checked on every run; much faster on a CPU).
* **Rhyme:** a written-form **heuristic** of the rawi (rhyme letter) at the end of each bayt. It is *not* the prosody verifier;
  read it next to the reference column, which shows how the same heuristic behaves on human poems.
* **Diacritics:** how much of the text carries harakat, which tells the verifier how much vowelisation it must infer.

Outputs: `MyDrive/shaer-prosody/m0/` (per-poem rows, per-system summaries, `m0_report.md`, `m0_summary.json`).

In [ ]:
if RUN_MO:
  pv.run_m0(pvc)

In [ ]:
if RUN_MO:
  pv.show_m0(pvc)

## 5. M1 - the prosody verifier

The verifier checks a poem against a requested meter and form, the way a prosodist would:

* **Rule tables** (`prosody/feet.py`, `prosody/meters.py`): the 8 base feet, every zihāf and ʿilla as an operation, the 16
  meters with their forms and ʿarūḍ/ḍarb tables. Every pattern is derived, never typed by hand.
* **Prosodic writing** (`prosody/arudi.py`): what is pronounced, not what is written (waṣl, sun letters, shadda, tanwīn,
  hidden long vowels, lengthened final vowels...). Text with few or no diacritics allows several readings; all of them are
  searched, and a pronunciation prior learned from the **train** split (`data/lexicon_train.json.gz`) says which readings
  are natural, so a line cannot be forced into a meter through an implausible reading.
* **Scansion** (`prosody/scan.py`) of every hemistich against every meter, with the nearest pattern and the broken foot
  when a line fails.
* **Rhyme** (`prosody/qafiya.py`): rawī, its vowel (iqwāʾ / iṣrāf), ridf, taʾsīs, waṣl, khurūj, muṭlaqa/muqayyada, īṭāʾ,
  and an Urdu-style refrain (radīf) when the poem has one.
* **Poem verdicts** (`prosody/verify.py`): allowed ʿarūḍ/ḍarb pairs, the same ḍarb (and ʿarūḍ) in every bayt, taṣrīʿ in the
  opening bayt, tadwīr, length.

Validation (CPU only): the unit tests; human poems of the **eval** split (tuning) and **test** split (report) as positives;
systematically broken copies as negatives; planted rhyme breaks; and bayt-by-bayt agreement with the BiLSTM classifier on
the released Shaer poems and on the human poems (needs M0). Everything is written to `MyDrive/shaer-prosody/m1/`.
`limit=0` runs a whole split (about 40 minutes per split on a 2-core CPU runtime).

In [ ]:
if RUN_M1:
  pv.m1_tests(pvc)                   # rule tables, prosodic writing, scansion, rhyme, poem verdicts

In [ ]:
if RUN_M1:
  LEXICON = pv.m1_lexicon(pvc)       # the shipped lexicon; pv.m1_lexicon(pvc, rebuild=True) rebuilds it from the train split
  pv.m1_validate(pvc, "eval", limit=1000)    # tuning split
  pv.m1_validate(pvc, "test", limit=1000)    # report split (limit=0: all 3,481 poems)

In [ ]:
if RUN_M1:
  pv.m1_compare_bilstm(pvc, limit=1000)     # verifier vs the BiLSTM on released Shaer poems and on human poems

In [ ]:
if RUN_M1:
  pv.m1_report(pvc)

**Try it on any poem.** `base_meter` and `form` use the Shaer labels (الطويل، الكامل ... / تام، مجزوء، مخلع، أحذ).
`ignore_diacritics=True` judges only the letters, so wrong written vowels do not count.

In [ ]:
if RUN_M1:
  poem = """قِفَا نَبْكِ مِنْ ذِكْرَى حَبِيبٍ وَمَنْزِلِ
  بِسِقْطِ اللِّوَى بَيْنَ الدَّخُولِ فَحَوْمَلِ
  فَتُوضِحَ فَالْمِقْرَاةِ لَمْ يَعْفُ رَسْمُهَا
  لِمَا نَسَجَتْهَا مِنْ جَنُوبٍ وَشَمْأَلِ"""
  report = pv.verify_poem(poem, "الطويل", requested_lines=4)

## 6. M3 - strict re-scoring of the baselines

Every system is re-scored with the verifier: the human reference poems, the released Shaer poems (E0), and every complete
baseline run on Drive (E1-E3). Shaer's own metrics (computed by the repository's `summarize_structural.py`, unchanged) are
shown next to the strict ones:

* **strictly valid poem**: every bayt in the requested meter and form, the ḍarb (and ʿarūḍ) form kept, the rhyme check
  passes, the exact number of hemistichs, no odd hemistich;
* meter, rhyme and length separately; bayts on meter; bayts with **no reading at all** in the requested meter (independent
  of the margin); the BiLSTM's per-bayt verdict on the same bayts;
* two reading modes: *letters* (written vowels ignored; the conservative meter verdict) and *binding* (written vowels
  count); margins 1.0, 1.5 (default), 3.0 and none;
* 95% bootstrap intervals over prompts, paired differences to the human poems, results by length, meter and bayt
  position, rhyme flaws by kind, pass@k when a run has several samples per prompt;
* two controls built from the human poems: each poem judged against the wrong meter, and poems whose halves come from
  two poems with different rhymes;
* integrity checks that must all be 0 (line counts against Shaer's parser, BiLSTM bayt alignment, margin recomputation,
  exact length against the official metric).

CPU runtime. Needs M0 (section 4). Each finished poem is written to `MyDrive/shaer-prosody/m3/` at once, so re-running a
cell resumes. The validation of this step is in `prosody-verifier/validation/M3_report.md`.

In [ ]:
if RUN_M3:
  pv.m3_rescore(pvc)        # all systems found on Drive; pv.m3_rescore(pvc, limit=200) for a quick look

  Drive: results are uploaded through the Drive API and verified (MD5)
  Drive → VM: m0: 14 file(s), 38.4 MB in 00:10
  Drive → VM: m3: 24 file(s), 56.6 MB in 00:18
  skipping e2_yehia_base: scored.jsonl has 318 of 3,481 rows (finish the baseline's scoring first)
systems: reference, e0_released_shaer, e1_shaer_adapter, e3_retrained_yehia

▶ M3 strict re-scoring   (expect 12 worker processes (one per CPU core); roughly 30-60 minutes per system and reading mode on a 2-core runtime, about 1/cores of that with more cores; resumable)
  reference/letters: 3,481 poems, 3,481 already scored, 0 to score
  reference/binding: 3,481 poems, 3,481 already scored, 0 to score
  e0_released_shaer/letters: 3,481 poems, 3,481 already scored, 0 to score
  e0_released_shaer/binding: 3,481 poems, 3,481 already scored, 0 to score
  e1_shaer_adapter/letters: 3,481 poems, 3,481 already scored, 0 to score
  e1_shaer_adapter/binding: 3,481 poems, 3,481 already scored, 0 to score
  e3_retrained_yehia/letters: 3,4

In [ ]:
if RUN_M3:
  pv.m3_report(pvc)

integrity checks: all 0


# M3 — strict re-scoring with the prosody verifier

Margin 1.5 unless stated; 95% bootstrap intervals over prompts (1000 resamples) in brackets. First sample per prompt. *letters*: written vowels ignored (the conservative meter verdict); *binding*: written vowels count.

**Strictly valid** = every bayt in the requested meter and form (ḍarb/ʿarūḍ form kept), rhyme check passes, exact number of hemistichs, no odd hemistich. *No reading*: share of bayts with a hemistich that cannot be read in the requested meter at all (any form, any cost; independent of the margin).

## Poems, letters

| system | poems | strictly valid | meter (form) | meter (any form) | rhyme | exact length |
|---|---:|---:|---:|---:|---:|---:|
| control_mixed_rhyme | 1,487 | 0.0 [0.0, 0.0] | 38.8 [36.3, 41.1] | 39.1 [36.5, 41.4] | 0.3 [0.1, 0.5] | 100.0 [100.0, 100.0] |
| control_wrong_meter | 3,481 | 0.0 [0.0, 0.0] | 0.0 [0.0, 0.0] | 0.0 [0.0, 0.1] | 93.1 [92.3, 93.9] | 100.0 [100.0, 100.0] |
| e0_released_shaer | 3,481 | 9.0 [8.0, 10.0] | 15.5 [14.3, 16.8] | 15.8 [14.5, 17.0] | 38.2 [36.6, 39.6] | 83.4 [82.2, 84.6] |
| e1_shaer_adapter | 3,481 | 5.2 [4.4, 5.9] | 8.9 [8.0, 9.8] | 9.1 [8.1, 10.1] | 36.3 [34.8, 37.8] | 82.1 [80.8, 83.4] |
| e3_retrained_yehia | 3,481 | 5.3 [4.5, 6.0] | 8.9 [8.0, 9.8] | 9.0 [8.2, 10.0] | 37.5 [35.9, 39.0] | 85.3 [84.2, 86.4] |
| reference | 3,481 | 54.9 [53.4, 56.5] | 57.1 [55.6, 58.7] | 59.4 [57.9, 61.1] | 93.1 [92.3, 93.9] | 100.0 [100.0, 100.0] |

## Bayts, letters

| system | bayts on meter (verifier) | any form | no reading in the meter | BiLSTM (Shaer metric) |
|---|---:|---:|---:|---:|
| control_mixed_rhyme | 81.6 [80.5, 82.7] | 85.7 [84.8, 86.6] | 5.4 [4.8, 6.0] | – |
| control_wrong_meter | 0.6 [0.4, 0.8] | 0.7 [0.5, 0.9] | 83.6 [82.5, 84.6] | – |
| e0_released_shaer | 53.8 [52.6, 54.8] | 54.8 [53.7, 55.8] | 25.8 [25.0, 26.6] | 95.4 [95.0, 95.8] |
| e1_shaer_adapter | 44.2 [42.9, 45.4] | 45.3 [44.1, 46.5] | 32.0 [31.0, 33.0] | 82.1 [81.1, 83.0] |
| e3_retrained_yehia | 41.9 [40.7, 43.0] | 43.2 [42.0, 44.3] | 31.3 [30.3, 32.3] | 79.6 [78.5, 80.6] |
| reference | 82.1 [81.0, 83.2] | 85.8 [85.0, 86.7] | 5.2 [4.6, 5.7] | 97.3 [96.7, 97.8] |

**Margin sensitivity (letters)**: strictly valid poems / bayts on meter

| system | margin 1.0 | margin 1.5 | margin 3.0 | margin none |
|---|---:|---:|---:|---:|
| control_mixed_rhyme | 0.0 / 79.9 | 0.0 / 81.6 | 0.1 / 86.6 | 0.1 / 88.5 |
| control_wrong_meter | 0.0 / 0.5 | 0.0 / 0.6 | 0.5 / 2.0 | 1.7 / 6.1 |
| e0_released_shaer | 8.4 / 51.7 | 9.0 / 53.8 | 11.9 / 61.3 | 13.6 / 65.5 |
| e1_shaer_adapter | 4.9 / 42.3 | 5.2 / 44.2 | 6.9 / 51.3 | 8.9 / 56.1 |
| e3_retrained_yehia | 4.6 / 39.6 | 5.3 / 41.9 | 7.8 / 49.8 | 9.6 / 55.0 |
| reference | 51.0 / 80.1 | 54.9 / 82.1 | 66.0 / 87.0 | 69.7 / 88.9 |

**Paired difference to the human reference poems (letters)**, percentage points:

| system | prompts | strictly valid | meter | rhyme | exact length | bayts on meter | no reading | BiLSTM bayts |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| control_mixed_rhyme | 1,487 | -43.4 [-45.7, -40.8] | -7.2 [-9.6, -4.8] | -90.5 [-91.9, -89.0] | 0.0 [0.0, 0.0] | -0.4 [-1.5, 0.7] | -0.1 [-0.8, 0.5] | – |
| control_wrong_meter | 3,481 | -54.9 [-56.5, -53.4] | -57.1 [-58.7, -55.6] | 0.0 [-0.1, 0.1] | 0.0 [0.0, 0.0] | -81.5 [-82.7, -80.3] | 78.4 [77.2, 79.5] | – |
| e0_released_shaer | 3,481 | -45.9 [-47.7, -44.2] | -41.6 [-43.3, -39.7] | -55.0 [-56.7, -53.2] | -16.6 [-17.8, -15.4] | -28.3 [-29.7, -26.9] | 20.6 [19.6, 21.7] | -1.9 [-2.6, -1.2] |
| e1_shaer_adapter | 3,481 | -49.8 [-51.4, -48.2] | -48.2 [-50.0, -46.4] | -56.8 [-58.5, -55.2] | -17.9 [-19.2, -16.6] | -37.9 [-39.4, -36.5] | 26.8 [25.7, 27.9] | -15.2 [-16.4, -14.2] |
| e3_retrained_yehia | 3,481 | -49.7 [-51.3, -48.0] | -48.2 [-49.9, -46.5] | -55.6 [-57.4, -54.0] | -14.7 [-15.8, -13.6] | -40.2 [-41.6, -38.9] | 26.1 [25.1, 27.1] | -17.7 [-18.9, -16.6] |

## Poems, binding

| system | poems | strictly valid | meter (form) | meter (any form) | rhyme | exact length |
|---|---:|---:|---:|---:|---:|---:|
| e0_released_shaer | 3,481 | 5.1 [4.3, 5.8] | 7.7 [6.8, 8.6] | 7.9 [7.0, 8.8] | 33.8 [32.4, 35.3] | 83.4 [82.2, 84.6] |
| e1_shaer_adapter | 3,481 | 3.1 [2.5, 3.7] | 5.2 [4.5, 6.0] | 5.3 [4.6, 6.1] | 32.2 [30.7, 33.6] | 82.1 [80.8, 83.4] |
| e3_retrained_yehia | 3,481 | 3.2 [2.6, 3.8] | 5.1 [4.4, 5.8] | 5.3 [4.6, 6.0] | 33.5 [32.0, 34.9] | 85.3 [84.2, 86.4] |
| reference | 3,481 | 52.5 [50.8, 54.1] | 54.7 [53.0, 56.3] | 57.2 [55.6, 58.9] | 92.3 [91.4, 93.2] | 100.0 [100.0, 100.0] |

## Bayts, binding

| system | bayts on meter (verifier) | any form | no reading in the meter | BiLSTM (Shaer metric) |
|---|---:|---:|---:|---:|
| e0_released_shaer | 40.4 [39.4, 41.4] | 41.3 [40.3, 42.3] | 50.1 [49.1, 51.2] | 95.4 [95.0, 95.8] |
| e1_shaer_adapter | 32.4 [31.3, 33.5] | 33.3 [32.3, 34.4] | 57.0 [55.8, 58.1] | 82.1 [81.1, 83.0] |
| e3_retrained_yehia | 30.5 [29.4, 31.6] | 31.5 [30.4, 32.5] | 58.5 [57.5, 59.7] | 79.6 [78.5, 80.6] |
| reference | 80.8 [79.7, 82.0] | 84.6 [83.8, 85.5] | 9.8 [9.1, 10.4] | 97.3 [96.7, 97.8] |

**Margin sensitivity (binding)**: strictly valid poems / bayts on meter

| system | margin 1.0 | margin 1.5 | margin 3.0 | margin none |
|---|---:|---:|---:|---:|
| e0_released_shaer | 4.8 / 39.8 | 5.1 / 40.4 | 6.0 / 43.2 | 6.5 / 44.9 |
| e1_shaer_adapter | 3.1 / 31.9 | 3.1 / 32.4 | 3.6 / 35.2 | 4.1 / 36.8 |
| e3_retrained_yehia | 2.9 / 29.9 | 3.2 / 30.5 | 3.9 / 33.3 | 4.3 / 35.1 |
| reference | 50.9 / 79.8 | 52.5 / 80.8 | 56.9 / 83.4 | 58.6 / 84.5 |

**Paired difference to the human reference poems (binding)**, percentage points:

| system | prompts | strictly valid | meter | rhyme | exact length | bayts on meter | no reading | BiLSTM bayts |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| e0_released_shaer | 3,481 | -47.4 [-49.1, -45.6] | -47.0 [-48.7, -45.2] | -58.5 [-60.2, -56.7] | -16.6 [-17.8, -15.4] | -40.4 [-42.0, -39.0] | 40.4 [39.2, 41.7] | -1.9 [-2.6, -1.2] |
| e1_shaer_adapter | 3,481 | -49.4 [-51.0, -47.5] | -49.5 [-51.2, -47.8] | -60.1 [-61.7, -58.5] | -17.9 [-19.2, -16.6] | -48.4 [-49.8, -47.0] | 47.2 [45.9, 48.5] | -15.2 [-16.4, -14.2] |
| e3_retrained_yehia | 3,481 | -49.3 [-51.0, -47.6] | -49.5 [-51.2, -47.8] | -58.8 [-60.6, -57.0] | -14.7 [-15.8, -13.6] | -50.3 [-51.8, -49.0] | 48.7 [47.5, 50.0] | -17.7 [-18.9, -16.6] |

## Shaer's own metrics (repository definitions) next to the strict ones (letters)

| system | meter conformity | base-meter accuracy | exact count | count adherence | strictly valid | meter (form) |
|---|---:|---:|---:|---:|---:|---:|
| e0_released_shaer | 0.9064 | 95.17% | 83.40% | 0.9792 | 9.0% | 15.5% |
| e1_shaer_adapter | 0.6613 | 80.15% | 82.07% | 0.9764 | 5.2% | 8.9% |
| e3_retrained_yehia | 0.6324 | 78.63% | 85.29% | 0.9833 | 5.3% | 8.9% |
| reference | 0.9482 | 96.93% | 100.00% | 1.0 | 54.9% | 57.1% |

## By requested length (letters)

| system | bayts | poems | strictly valid | meter | rhyme | exact length | bayts on meter | BiLSTM bayts |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| control_mixed_rhyme | 4-6 | 805 | 0.0 | 47.2 | 0.37 | 100.0 | 80.78 | – |
| control_mixed_rhyme | 7-10 | 419 | 0.0 | 34.84 | 0.0 | 100.0 | 82.01 | – |
| control_mixed_rhyme | 11-20 | 263 | 0.0 | 19.39 | 0.38 | 100.0 | 82.18 | – |
| control_wrong_meter | 1-3 | 1,668 | 0.0 | 0.0 | 96.52 | 100.0 | 0.51 | – |
| control_wrong_meter | 4-6 | 805 | 0.0 | 0.0 | 92.42 | 100.0 | 0.34 | – |
| control_wrong_meter | 7-10 | 473 | 0.0 | 0.0 | 91.97 | 100.0 | 0.51 | – |
| control_wrong_meter | 11-20 | 535 | 0.0 | 0.0 | 84.67 | 100.0 | 0.73 | – |
| e0_released_shaer | 1-3 | 1,668 | 17.51 | 27.22 | 57.37 | 98.44 | 49.06 | 97.01 |
| e0_released_shaer | 4-6 | 805 | 2.24 | 8.2 | 31.06 | 93.17 | 50.65 | 94.99 |
| e0_released_shaer | 7-10 | 473 | 0.63 | 3.38 | 16.91 | 75.26 | 54.46 | 94.29 |
| e0_released_shaer | 11-20 | 535 | 0.0 | 0.75 | 7.85 | 28.97 | 56.88 | 95.36 |
| e1_shaer_adapter | 1-3 | 1,668 | 10.13 | 16.01 | 53.84 | 97.66 | 36.85 | 77.76 |
| e1_shaer_adapter | 4-6 | 805 | 1.24 | 4.35 | 30.56 | 92.55 | 40.51 | 79.91 |
| e1_shaer_adapter | 7-10 | 473 | 0.21 | 1.06 | 16.49 | 74.0 | 43.12 | 81.23 |
| e1_shaer_adapter | 11-20 | 535 | 0.0 | 0.56 | 7.85 | 24.86 | 49.42 | 85.23 |
| e3_retrained_yehia | 1-3 | 1,668 | 10.19 | 16.37 | 54.56 | 98.44 | 37.21 | 76.37 |
| e3_retrained_yehia | 4-6 | 805 | 1.61 | 3.6 | 33.42 | 92.67 | 39.84 | 77.75 |
| e3_retrained_yehia | 7-10 | 473 | 0.0 | 0.85 | 16.28 | 81.18 | 42.2 | 80.12 |
| e3_retrained_yehia | 11-20 | 535 | 0.0 | 0.56 | 9.35 | 36.82 | 44.72 | 81.6 |
| reference | 1-3 | 1,668 | 70.5 | 72.3 | 96.58 | 100.0 | 81.77 | 97.24 |
| reference | 4-6 | 805 | 50.19 | 52.67 | 92.42 | 100.0 | 80.3 | 96.44 |
| reference | 7-10 | 473 | 41.44 | 43.55 | 91.75 | 100.0 | 82.13 | 97.17 |
| reference | 11-20 | 535 | 25.42 | 28.22 | 84.67 | 100.0 | 83.08 | 97.8 |

## Bayts on meter by position in the poem (letters)

| system | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10+ |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| control_mixed_rhyme | 82.45 | 81.44 | 82.18 | 83.05 | 81.69 | 81.84 | 82.26 | 80.23 | 81.86 | 78.9 |
| control_wrong_meter | 0.6 | 0.49 | 0.62 | 0.44 | 0.35 | 0.51 | 0.79 | 0.7 | 0.27 | 0.72 |
| e0_released_shaer | 53.17 | 51.87 | 51.5 | 54.18 | 54.55 | 54.77 | 54.3 | 56.68 | 55.43 | 55.42 |
| e1_shaer_adapter | 42.09 | 39.52 | 42.29 | 43.84 | 45.36 | 47.4 | 45.56 | 45.76 | 44.47 | 49.03 |
| e3_retrained_yehia | 39.47 | 40.75 | 41.69 | 42.83 | 42.85 | 42.86 | 43.01 | 43.46 | 44.75 | 42.94 |
| reference | 82.1 | 81.92 | 82.27 | 81.91 | 83.0 | 81.86 | 82.24 | 82.51 | 83.38 | 81.48 |

## Rhyme and form issues (letters, share of poems)

| system | arud_changes | count | darb_changes | iqwa | israf | ita | mutlaqa_muqayyada | odd_tail | radif_missing | rawi | sinad_ridf | sinad_tasis |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| control_mixed_rhyme | 5.78 | 0.0 | 10.56 | 1.41 | 7.87 | 3.7 | 0.13 | 0.0 | 0.0 | 99.66 | 48.22 | 19.84 |
| control_wrong_meter | 1.18 | 0.0 | 1.21 | 0.03 | 0.57 | 4.22 | 0.09 | 0.0 | 0.14 | 3.48 | 3.33 | 2.59 |
| e0_released_shaer | 1.75 | 16.6 | 8.07 | 2.04 | 8.68 | 7.7 | 0.23 | 2.36 | 0.78 | 52.14 | 29.91 | 12.96 |
| e1_shaer_adapter | 1.9 | 17.93 | 8.93 | 2.1 | 7.93 | 7.41 | 0.49 | 2.7 | 0.63 | 54.04 | 34.76 | 13.16 |
| e3_retrained_yehia | 1.84 | 14.71 | 8.85 | 1.49 | 7.44 | 8.24 | 0.75 | 2.84 | 0.46 | 53.29 | 31.66 | 13.65 |
| reference | 1.75 | 0.0 | 1.41 | 0.03 | 0.66 | 4.22 | 0.09 | 0.0 | 0.14 | 3.48 | 3.27 | 2.61 |

## By meter (letters; strictly valid / bayts on meter / BiLSTM bayts)

| meter | control_mixed_rhyme | control_wrong_meter | e0_released_shaer | e1_shaer_adapter | e3_retrained_yehia | reference |
|---|---|---|---|---|---|---|
| tawil | 0.0 / 89.19 / – (352) | 0.0 / 0.68 / – (191) | 14.63 / 71.43 / 99.96 (841) | 9.27 / 64.7 / 95.73 (841) | 9.27 / 62.27 / 94.67 (841) | 65.64 / 89.65 / 98.2 (841) |
| kamil | 0.0 / 80.37 / – (284) | 0.0 / 4.33 / – (199) | 7.4 / 53.12 / 95.04 (649) | 3.24 / 40.6 / 79.8 (649) | 4.01 / 41.28 / 79.86 (649) | 59.48 / 82.7 / 97.74 (649) |
| basit | 0.0 / 84.93 / – (230) | 0.0 / 1.17 / – (173) | 9.07 / 56.78 / 98.29 (529) | 6.62 / 45.26 / 84.44 (529) | 6.43 / 36.53 / 74.92 (529) | 61.25 / 84.43 / 97.75 (529) |
| wafir | 0.0 / 86.09 / – (149) | 0.0 / 0.0 / – (209) | 9.41 / 52.31 / 96.54 (340) | 5.0 / 43.15 / 82.43 (340) | 4.71 / 41.49 / 82.61 (340) | 60.88 / 86.95 / 94.93 (340) |
| khafif | 0.0 / 84.04 / – (141) | 0.0 / 0.19 / – (188) | 7.8 / 50.28 / 98.22 (295) | 4.75 / 39.0 / 85.92 (295) | 2.71 / 34.95 / 82.21 (295) | 50.85 / 83.43 / 97.61 (295) |
| mujtath | 0.0 / 80.0 / – (24) | 0.0 / 0.14 / – (255) | 4.92 / 52.51 / 86.96 (61) | 3.28 / 38.85 / 68.47 (61) | 1.64 / 42.18 / 62.59 (61) | 57.38 / 81.97 / 95.92 (61) |
| mudari | – | 0.0 / 0.0 / – (245) | – | – | – | – |
| rajaz | 0.0 / 59.73 / – (53) | 0.0 / 1.06 / – (244) | 5.84 / 31.9 / 84.24 (137) | 2.19 / 27.23 / 62.5 (137) | 2.19 / 21.88 / 58.38 (137) | 27.01 / 57.95 / 94.44 (137) |
| madid | 0.0 / 70.77 / – (9) | 0.0 / 0.22 / – (241) | 0.0 / 37.89 / 65.26 (17) | 0.0 / 20.0 / 40.0 (17) | 0.0 / 17.02 / 31.91 (17) | 41.18 / 75.27 / 94.62 (17) |
| muqtadab | – | 0.0 / 0.0 / – (230) | – | – | – | – |
| mutaqarib | 0.0 / 69.92 / – (58) | 0.0 / 0.34 / – (229) | 7.69 / 44.41 / 96.15 (156) | 1.92 / 34.92 / 83.04 (156) | 5.13 / 37.09 / 82.67 (156) | 33.97 / 71.45 / 97.39 (156) |
| munsarih | 0.0 / 80.08 / – (37) | 0.0 / 0.47 / – (219) | 1.19 / 34.85 / 82.0 (84) | 1.19 / 26.1 / 57.74 (84) | 2.38 / 28.64 / 57.04 (84) | 39.29 / 78.54 / 97.17 (84) |
| hazaj | 0.0 / 73.58 / – (9) | 0.0 / 0.0 / – (218) | 5.56 / 31.73 / 80.77 (18) | 0.0 / 16.07 / 51.79 (18) | 0.0 / 21.3 / 57.41 (18) | 44.44 / 68.93 / 92.23 (18) |
| sari | 0.0 / 76.46 / – (78) | 0.0 / 0.27 / – (211) | 5.09 / 34.86 / 87.71 (216) | 1.85 / 23.49 / 63.58 (216) | 2.31 / 24.0 / 60.06 (216) | 39.35 / 74.54 / 96.88 (216) |
| mutadarik | – | 0.0 / 0.35 / – (215) | – | – | – | – |
| ramal | 0.0 / 57.45 / – (63) | 0.0 / 0.34 / – (214) | 2.17 / 28.1 / 86.84 (138) | 1.45 / 25.06 / 65.56 (138) | 1.45 / 24.53 / 65.93 (138) | 25.36 / 63.47 / 97.71 (138) |

## Integrity checks (all must be 0)

| system/mode | line counts differing from Shaer's | BiLSTM bayts misaligned | margin recomputation mismatches | exact length vs official (pp) | errors |
|---|---:|---:|---:|---:|---:|
| control_mixed_rhyme/letters | 0 | 0 | 0 | – | 0 |
| control_wrong_meter/letters | 0 | 0 | 0 | – | 0 |
| e0_released_shaer/letters | 0 | 0 | 0 | 0.0 | 0 |
| e0_released_shaer/binding | 0 | 0 | 0 | 0.0 | 0 |
| e1_shaer_adapter/letters | 0 | 0 | 0 | 0.0 | 0 |
| e1_shaer_adapter/binding | 0 | 0 | 0 | 0.0 | 0 |
| e3_retrained_yehia/letters | 0 | 0 | 0 | 0.0 | 0 |
| e3_retrained_yehia/binding | 0 | 0 | 0 | 0.0 | 0 |
| reference/letters | 0 | 0 | 0 | 0.0 | 0 |
| reference/binding | 0 | 0 | 0 | 0.0 | 0 |


  ☁ Drive: everything this step wrote is on Drive (0 upload(s), the last 0)


## 7. M4 - verifier-guided decoding against best-of-N, at equal compute

Two ways of spending more samples on one prompt, both with the released Shaer adapter on Yehia-7B-preview and the paper's
decoding (temperature 0.55, top-p 0.90, repetition penalty 1.08):

* **I0 / I1, best-of-N:** sample N whole poems; keep the first strictly valid one (letters mode, margin 1.5), else the one
  with the fewest defects. I0 is N = 1, the paper's setting.
* **I2, hemistich search:** write the poem one hemistich at a time; for each, sample k candidates (up to R rounds) and keep
  the first that adds no new defect to the poem so far (meter, ḍarb, rhyme with the rawī fixed by the first bayt, no
  repeated rhyme word); else the one adding the fewest. Exactly the requested number of hemistichs is written.

Both keep the model's own first acceptable sample, so the verifier filters but never picks the "most metrical" line.
The budget is every generated token, rejected candidates included. Each search run is compared with best-of-N at the
smallest N that used at least as many tokens, with paired bootstrap intervals and an exact McNemar test. The selecting
verifier is also the scorer, so the report also shows the BiLSTM metric of the paper, binding mode and no margin, plus
repetition, diversity and copying from the training poems.

**Order** (each step resumes; every finished poem is on Drive under `MyDrive/shaer-prosody/m4/`):

1. `m4_selfcheck`: a few minutes on the GPU; it must say PASSED before the long runs.
2. `m4_dev`: the development sweep on 400 seeded eval prompts, on the GPU. It makes one run of 16 samples per prompt
   (the long one, about 2 h on an A100) and five search runs (minutes to tens of minutes each). Each run loads the model
   again (a few minutes).
3. `m4_score` and `m4_report` on `"eval"`: a CPU runtime is enough. The report prints the test configuration chosen by the
   rule fixed in advance: the search run with the highest strict validity among those within the budget of best-of-4.
4. `m4_test_from_choice`: the chosen search run and the matching best-of-N on the 3,481 test prompts (GPU). Best-of-N
   takes N times one sample's time, about 1.5 h per sample on an A100.
5. `m4_score` and `m4_report` on `"test"`.

`m4_check_sampling` (optional, about 20 min on the GPU) checks that the sample mode reproduces the baseline's E1
decoding on 300 test prompts. An L4 works too, with smaller batches (about 3-5 times slower). The validation is in
`prosody-verifier/validation/M4_report.md`.

In [ ]:
if RUN_M4:
  pv.m4_selfcheck(pvc)          # GPU: shared cache vs full forward pass; stepwise vs one-shot greedy decoding

In [ ]:
if RUN_M4:
  pv.m4_dev(pvc)                # GPU: development sweep on the eval split; pv.m4_dev(pvc, only=("i2_k4_r1",)) runs one

  Drive → VM: m4/eval: 63 file(s), 119.9 MB in 00:55

▶ M4 development sweep: 7 runs on 400 eval prompts   (expect the sample run is the long one (about 2 h on an A100 for 16 samples per prompt); each search run takes minutes to tens of minutes; every finished poem is on Drive)
  Generating train split         [░░░░░░░░░░░░░░░░░░░░]   0%  0/109070  elapsed 00:00
  Generating train split         [████░░░░░░░░░░░░░░░░]  20%  22000/109070  elapsed 00:00
  Generating train split         [██████░░░░░░░░░░░░░░]  30%  33000/109070  elapsed 00:00  (102,690.1 it/s)  ETA 00:00
  Generating train split         [████████░░░░░░░░░░░░]  40%  44000/109070  elapsed 00:00  (102,690.1 it/s)  ETA 00:00
  Generating train split         [██████████░░░░░░░░░░]  51%  55535/109070  elapsed 00:00  (102,690.1 it/s)  ETA 00:00
  Generating train split         [████████████░░░░░░░░]  62%  67535/109070  elapsed 00:00  (103,062.0 it/s)  ETA 00:00
  Generating train split         [███████████████░░░░░]  73%  79535/1

In [7]:
if RUN_M4:
  pv.m4_score(pvc, "eval")      # CPU is enough: the repository's scorer and the M3 verifier on every run
  pv.m4_report(pvc, "eval")
  pv.m4_export(pvc, "eval")    # every poem with its prompt, scores and accounting: m4/eval/export/*.csv (open in Sheets)

  Drive: results are uploaded through the Drive API and verified (MD5)
  Drive → VM: m4: 70 file(s), 269.0 MB in 00:21
  i1_n16/letters: 6,400 poems, 6,400 already scored, 0 to score
  i1_n16/binding: 6,400 poems, 6,400 already scored, 0 to score
  i2_k1_r4/letters: 400 poems, 400 already scored, 0 to score
  i2_k1_r4/binding: 400 poems, 400 already scored, 0 to score
  i2_k1_r8/letters: 400 poems, 400 already scored, 0 to score
  i2_k1_r8/binding: 400 poems, 400 already scored, 0 to score
  i2_k4_r1/letters: 400 poems, 400 already scored, 0 to score
  i2_k4_r1/binding: 400 poems, 400 already scored, 0 to score
  i2_k4_r2/letters: 400 poems, 400 already scored, 0 to score
  i2_k4_r2/binding: 400 poems, 400 already scored, 0 to score
  i2_k4_r2_b/letters: 400 poems, 400 already scored, 0 to score
  i2_k4_r2_b/binding: 400 poems, 400 already scored, 0 to score
  i2_k8_r1/letters: 400 poems, 400 already scored, 0 to score
  i2_k8_r1/binding: 400 poems, 400 already scored, 0 to score
  ☁ D

# M4 — verifier-guided decoding against best-of-N

400 prompts shared by every condition; letters mode, margin 1.5; 95% bootstrap intervals over prompts (1000 resamples). Budget = generated tokens per prompt, rejected candidates included.

**Pre-registered choice for the test runs** (search runs within the budget of best-of-4, highest strict validity): I2 k=1 R=8 [i2_k1_r8] (k = 1, R = 8), against best-of-4.

## Results

| condition | gen tokens / prompt | encoded context tokens / prompt | GPU s / prompt | strictly valid | valid ignoring length | meter | rhyme | exact length | bayts on meter | bayts with no reading | BiLSTM bayts |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| I0 single sample [i1_n16] | 157 | 181 | 2.15 | 5.5 [3.5, 8.0] | 5.5 [3.5, 8.0] | 10.5 [7.5, 13.8] | 38.8 [34.2, 43.8] | 82.2 [78.5, 85.8] | 38.4 [35.0, 41.8] | 35.0 [31.8, 38.5] | 78.2 [75.0, 81.4] |
| I2 k=1 R=4 [i2_k1_r4] | 271 | 3,386 | 9.36 | 45.0 [40.5, 50.0] | 45.0 [40.5, 50.0] | 57.5 [52.8, 62.5] | 73.0 [68.7, 77.5] | 100.0 [100.0, 100.0] | 87.0 [84.8, 89.1] | 3.3 [2.4, 4.3] | 91.8 [89.8, 93.6] |
| I1 best-of-2 [i1_n16] | 308 | 362 | 4.28 | 11.0 [8.0, 14.5] | 11.0 [8.0, 14.5] | 16.8 [13.0, 20.8] | 51.8 [46.7, 56.5] | 86.0 [82.5, 89.2] | 49.6 [46.4, 53.0] | 26.8 [24.2, 29.3] | 84.8 [82.2, 87.3] |
| I2 k=1 R=8 [i2_k1_r8] | 318 | 3,399 | 11.75 | 66.5 [62.0, 71.2] | 66.5 [62.0, 71.2] | 80.8 [77.0, 84.2] | 80.8 [76.8, 84.8] | 100.0 [100.0, 100.0] | 95.8 [94.9, 96.7] | 1.1 [0.7, 1.6] | 94.7 [92.8, 96.2] |
| I2 k=4 R=1 [i2_k4_r1] | 586 | 3,366 | 10.25 | 41.2 [36.2, 46.0] | 41.2 [36.2, 46.0] | 59.8 [54.5, 64.5] | 69.0 [64.8, 73.3] | 100.0 [100.0, 100.0] | 87.8 [85.7, 89.8] | 3.3 [2.5, 4.2] | 92.7 [90.8, 94.3] |
| I1 best-of-4 [i1_n16] | 614 | 724 | 8.48 | 16.0 [12.5, 19.8] | 16.0 [12.5, 19.8] | 22.0 [18.0, 26.2] | 60.5 [55.5, 65.2] | 88.5 [85.0, 91.5] | 56.9 [54.0, 59.9] | 21.9 [20.0, 23.9] | 87.3 [85.2, 89.4] |
| I2 k=4 R=2 [i2_k4_r2] | 656 | 3,371 | 11.56 | 68.0 [63.5, 72.2] | 68.0 [63.5, 72.2] | 81.0 [77.0, 85.0] | 81.8 [78.0, 85.2] | 100.0 [100.0, 100.0] | 94.8 [93.0, 96.2] | 1.7 [0.8, 3.1] | 93.7 [92.1, 95.1] |
| I2 k=4 R=2 binding [i2_k4_r2_b] | 673 | 3,333 | 11.85 | 41.0 [36.2, 46.0] | 41.0 [36.2, 46.0] | 50.8 [46.0, 55.5] | 78.5 [74.5, 82.5] | 100.0 [100.0, 100.0] | 85.4 [83.4, 87.4] | 2.9 [2.1, 3.8] | 93.5 [91.7, 95.1] |
| I2 k=8 R=1 [i2_k8_r1] | 1,196 | 3,415 | 15.43 | 68.2 [63.5, 72.8] | 68.2 [63.5, 72.8] | 78.2 [74.0, 82.2] | 83.5 [79.5, 86.8] | 100.0 [100.0, 100.0] | 94.9 [93.4, 96.2] | 1.2 [0.8, 1.7] | 93.7 [91.9, 95.2] |
| I1 best-of-8 [i1_n16] | 1,225 | 1,448 | 16.76 | 25.5 [21.2, 29.8] | 25.8 [21.5, 30.2] | 31.8 [27.2, 36.5] | 65.2 [60.5, 69.8] | 91.0 [88.0, 93.8] | 64.5 [61.6, 67.2] | 18.6 [16.8, 20.5] | 87.8 [85.8, 90.0] |
| I1 best-of-16 [i1_n16] | 2,457 | 2,896 | 33.69 | 36.8 [32.2, 41.8] | 36.8 [32.2, 41.8] | 42.0 [37.0, 46.8] | 70.0 [65.5, 74.5] | 92.2 [89.5, 94.8] | 71.6 [68.8, 74.1] | 14.6 [12.9, 16.4] | 88.5 [86.0, 90.6] |

**Independent views** (the verifier that selects is also the one that scores above):

| condition | Shaer meter conformity | base-meter accuracy | strictly valid, no margin | strictly valid, binding | meter, binding |
|---|---:|---:|---:|---:|---:|
| I0 single sample [i1_n16] | 0.635 | 78.5 | 8.2 [5.8, 11.2] | 2.5 [1.0, 4.2] | 5.0 [3.0, 7.2] |
| I2 k=1 R=4 [i2_k1_r4] | 0.8139 | 87.5 | 58.2 [53.2, 63.0] | 14.8 [11.5, 18.5] | 20.5 [16.7, 24.8] |
| I1 best-of-2 [i1_n16] | 0.6972 | 85.0 | 15.2 [11.8, 19.0] | 4.2 [2.5, 6.2] | 6.0 [3.8, 8.5] |
| I2 k=1 R=8 [i2_k1_r8] | 0.8739 | 94.5 | 72.2 [68.0, 76.8] | 20.8 [17.0, 25.0] | 27.8 [23.8, 32.5] |
| I2 k=4 R=1 [i2_k4_r1] | 0.8306 | 91.0 | 52.0 [47.0, 56.8] | 17.2 [13.5, 20.8] | 24.8 [20.5, 28.8] |
| I1 best-of-4 [i1_n16] | 0.7398 | 87.75 | 21.5 [17.7, 25.5] | 6.8 [4.2, 9.3] | 8.5 [5.8, 11.2] |
| I2 k=4 R=2 [i2_k4_r2] | 0.8595 | 91.5 | 75.0 [71.0, 79.0] | 22.5 [18.5, 26.8] | 29.2 [25.0, 34.0] |
| I2 k=4 R=2 binding [i2_k4_r2_b] | 0.855 | 89.0 | 60.8 [56.0, 65.5] | 50.5 [45.5, 55.3] | 64.5 [60.0, 69.0] |
| I2 k=8 R=1 [i2_k8_r1] | 0.8552 | 91.0 | 75.2 [70.8, 79.2] | 20.0 [16.5, 24.0] | 23.5 [19.5, 27.8] |
| I1 best-of-8 [i1_n16] | 0.7533 | 86.5 | 30.0 [25.5, 34.5] | 10.0 [7.0, 13.0] | 11.5 [8.2, 14.8] |
| I1 best-of-16 [i1_n16] | 0.775 | 88.5 | 43.0 [38.2, 47.8] | 13.8 [10.2, 17.0] | 16.0 [12.5, 19.5] |

## Matched compute: each search run against best-of-N

| search run | against | budget ratio (I1 / I2) | strictly valid | valid ignoring length | meter | rhyme | BiLSTM bayts | McNemar (I2 only / I1 only, p) |
|---|---|---:|---:|---:|---:|---:|---:|---|
| I2 k=1 R=4 [i2_k1_r4] | I1 best-of-2 [i1_n16] | 1.136 | 34.0 [28.8, 39.2] | 34.0 [28.8, 39.2] | 40.8 [35.0, 46.5] | 21.2 [15.2, 27.5] | 7.0 [4.7, 9.5] | 152 / 16, p = 5.5e-29 |
| I2 k=1 R=4 [i2_k1_r4] | I0 single sample [i1_n16] | 0.577 | 39.5 [35.0, 44.8] | 39.5 [35.0, 44.8] | 47.0 [41.5, 52.5] | 34.2 [28.2, 40.2] | 13.6 [10.3, 16.9] | 166 / 8, p = 1.6e-39 |
| I2 k=1 R=8 [i2_k1_r8] | I1 best-of-4 [i1_n16] | 1.932 | 50.5 [45.5, 56.0] | 50.5 [45.5, 56.0] | 58.8 [53.5, 63.8] | 20.2 [14.0, 26.0] | 7.3 [5.6, 9.2] | 212 / 10, p = 2e-50 |
| I2 k=1 R=8 [i2_k1_r8] | I1 best-of-2 [i1_n16] | 0.969 | 55.5 [50.2, 60.8] | 55.5 [50.2, 60.8] | 64.0 [59.0, 68.8] | 29.0 [22.8, 35.2] | 9.8 [7.7, 12.1] | 230 / 8, p = 1.1e-57 |
| I2 k=4 R=1 [i2_k4_r1] | I1 best-of-4 [i1_n16] | 1.048 | 25.2 [19.8, 30.0] | 25.2 [19.8, 30.0] | 37.8 [32.0, 43.5] | 8.5 [2.5, 14.5] | 5.4 [3.5, 7.4] | 124 / 23, p = 6.1e-18 |
| I2 k=4 R=1 [i2_k4_r1] | I1 best-of-2 [i1_n16] | 0.526 | 30.2 [24.8, 35.2] | 30.2 [24.8, 35.2] | 43.0 [37.2, 48.8] | 17.2 [11.0, 23.3] | 7.9 [5.6, 10.4] | 137 / 16, p = 3.8e-25 |
| I2 k=4 R=2 [i2_k4_r2] | I1 best-of-8 [i1_n16] | 1.866 | 42.5 [36.5, 48.5] | 42.2 [36.2, 48.0] | 49.2 [44.0, 54.5] | 16.5 [11.5, 22.0] | 5.9 [4.1, 7.7] | 187 / 17, p = 2.2e-37 |
| I2 k=4 R=2 [i2_k4_r2] | I1 best-of-4 [i1_n16] | 0.936 | 52.0 [46.7, 57.5] | 52.0 [46.7, 57.5] | 59.0 [53.8, 64.5] | 21.2 [16.0, 27.0] | 6.4 [4.6, 8.3] | 218 / 10, p = 4.2e-52 |
| I2 k=4 R=2 binding [i2_k4_r2_b] | I1 best-of-8 [i1_n16] | 1.819 | 15.5 [10.0, 20.8] | 15.2 [9.8, 20.5] | 19.0 [13.5, 24.8] | 13.2 [7.5, 19.2] | 5.7 [4.1, 7.4] | 100 / 38, p = 1.3e-07 |
| I2 k=4 R=2 binding [i2_k4_r2_b] | I1 best-of-4 [i1_n16] | 0.912 | 25.0 [19.5, 30.5] | 25.0 [19.5, 30.5] | 28.8 [22.8, 34.2] | 18.0 [12.2, 24.2] | 6.2 [4.5, 8.2] | 125 / 25, p = 3.4e-17 |
| I2 k=8 R=1 [i2_k8_r1] | I1 best-of-8 [i1_n16] | 1.024 | 42.8 [37.0, 48.3] | 42.5 [37.0, 47.8] | 46.5 [40.8, 52.2] | 18.2 [12.5, 24.0] | 5.9 [3.9, 7.8] | 189 / 18, p = 3.8e-37 |
| I2 k=8 R=1 [i2_k8_r1] | I1 best-of-4 [i1_n16] | 0.514 | 52.2 [47.0, 57.5] | 52.2 [47.0, 57.5] | 56.2 [50.8, 61.8] | 23.0 [17.5, 28.8] | 6.4 [4.5, 8.4] | 219 / 10, p = 2.2e-52 |

## Against the single sample (I0), paired, percentage points

| condition | strictly valid | valid ignoring length | meter | rhyme | BiLSTM bayts |
|---|---:|---:|---:|---:|---:|
| I1 best-of-2 [i1_n16] | 5.5 [3.2, 7.8] | 5.5 [3.2, 7.8] | 6.2 [3.8, 9.0] | 13.0 [9.0, 16.5] | 6.6 [4.0, 9.5] |
| I1 best-of-4 [i1_n16] | 10.5 [7.2, 13.8] | 10.5 [7.2, 13.8] | 11.5 [8.2, 15.0] | 21.8 [16.8, 26.5] | 9.1 [6.0, 12.4] |
| I1 best-of-8 [i1_n16] | 20.0 [16.5, 24.0] | 20.2 [16.5, 24.2] | 21.2 [17.2, 25.8] | 26.5 [21.2, 31.5] | 9.6 [6.3, 12.9] |
| I1 best-of-16 [i1_n16] | 31.2 [26.5, 35.8] | 31.2 [26.5, 35.8] | 31.5 [26.8, 36.0] | 31.2 [25.5, 36.3] | 10.2 [6.8, 13.7] |
| I2 k=1 R=4 [i2_k1_r4] | 39.5 [35.0, 44.8] | 39.5 [35.0, 44.8] | 47.0 [41.5, 52.5] | 34.2 [28.2, 40.2] | 13.6 [10.3, 16.9] |
| I2 k=1 R=8 [i2_k1_r8] | 61.0 [56.2, 65.8] | 61.0 [56.2, 65.8] | 70.2 [65.8, 75.0] | 42.0 [35.5, 47.5] | 16.4 [13.2, 19.8] |
| I2 k=4 R=1 [i2_k4_r1] | 35.8 [30.7, 40.5] | 35.8 [30.7, 40.5] | 49.2 [43.8, 54.8] | 30.2 [24.0, 35.8] | 14.5 [11.1, 17.9] |
| I2 k=4 R=2 [i2_k4_r2] | 62.5 [57.5, 67.2] | 62.5 [57.5, 67.2] | 70.5 [65.7, 75.0] | 43.0 [37.5, 48.3] | 15.4 [12.2, 18.9] |
| I2 k=4 R=2 binding [i2_k4_r2_b] | 35.5 [30.5, 40.3] | 35.5 [30.5, 40.3] | 40.2 [35.0, 45.2] | 39.8 [34.0, 45.5] | 15.3 [11.9, 18.7] |
| I2 k=8 R=1 [i2_k8_r1] | 62.8 [57.8, 67.5] | 62.8 [57.8, 67.5] | 67.8 [62.5, 72.5] | 44.8 [38.8, 50.2] | 15.4 [12.1, 19.0] |

## By requested length (strictly valid / valid ignoring length / gen tokens per prompt)

| condition | 1-3 bayts | 4-6 bayts | 7-10 bayts | 11-20 bayts |
|---|---:|---:|---:|---:|
| I0 single sample [i1_n16] | 9.13 / 9.13 / 57 | 3.8 / 3.8 / 134 | 0.0 / 0.0 / 262 | 0.0 / 0.0 / 428 |
| I2 k=1 R=4 [i2_k1_r4] | 50.96 / 50.96 / 118 | 41.77 / 41.77 / 255 | 30.61 / 30.61 / 461 | 40.62 / 40.62 / 643 |
| I1 best-of-2 [i1_n16] | 19.71 / 19.71 / 114 | 3.8 / 3.8 / 269 | 0.0 / 0.0 / 510 | 0.0 / 0.0 / 833 |
| I2 k=1 R=8 [i2_k1_r8] | 69.23 / 69.23 / 149 | 59.49 / 59.49 / 301 | 61.22 / 61.22 / 514 | 70.31 / 70.31 / 739 |
| I2 k=4 R=1 [i2_k4_r1] | 54.33 / 54.33 / 228 | 30.38 / 30.38 / 516 | 22.45 / 22.45 / 1000 | 26.56 / 26.56 / 1520 |
| I1 best-of-4 [i1_n16] | 28.37 / 28.37 / 233 | 6.33 / 6.33 / 533 | 0.0 / 0.0 / 1004 | 0.0 / 0.0 / 1656 |
| I2 k=4 R=2 [i2_k4_r2] | 70.67 / 70.67 / 271 | 63.29 / 63.29 / 625 | 67.35 / 67.35 / 1035 | 65.62 / 65.62 / 1656 |
| I2 k=4 R=2 binding [i2_k4_r2_b] | 47.12 / 47.12 / 287 | 37.97 / 37.97 / 641 | 28.57 / 28.57 / 1038 | 34.38 / 34.38 / 1691 |
| I2 k=8 R=1 [i2_k8_r1] | 69.71 / 69.71 / 465 | 68.35 / 68.35 / 1069 | 65.31 / 65.31 / 1906 | 65.62 / 65.62 / 3184 |
| I1 best-of-8 [i1_n16] | 43.27 / 43.27 / 466 | 13.92 / 13.92 / 1060 | 2.04 / 2.04 / 1982 | 0.0 / 1.56 / 3317 |
| I1 best-of-16 [i1_n16] | 62.02 / 62.02 / 932 | 21.52 / 21.52 / 2130 | 2.04 / 2.04 / 3929 | 0.0 / 0.0 / 6688 |

## Quality and copying

| condition | words_per_hemistich | distinct_1 | distinct_2 | distinct_2_within_poem | repeated_hemistichs_pct | distinct_rhyme_words_pct | copied_from_reference_pct | copied_from_train_pct |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| I0 single sample [i1_n16] | 4.87 | 44.7 | 91.79 | 99.01 | 1.39 | 97.38 | 0.18 | 0.16 |
| I2 k=1 R=4 [i2_k1_r4] | 4.9 | 44.41 | 92.28 | 99.34 | 0.19 | 99.16 | 0.47 | 0.16 |
| I1 best-of-2 [i1_n16] | 4.88 | 44.94 | 92.08 | 99.21 | 1.13 | 97.51 | 0.23 | 0.14 |
| I2 k=1 R=8 [i2_k1_r8] | 4.93 | 44.61 | 92.73 | 99.36 | 0.07 | 99.34 | 0.47 | 0.14 |
| I2 k=4 R=1 [i2_k4_r1] | 4.9 | 44.71 | 92.53 | 99.3 | 0.38 | 99.2 | 0.28 | 0.16 |
| I1 best-of-4 [i1_n16] | 4.91 | 45.19 | 92.97 | 99.47 | 0.14 | 98.31 | 0.41 | 0.16 |
| I2 k=4 R=2 [i2_k4_r2] | 4.91 | 44.58 | 92.36 | 99.31 | 0.38 | 99.16 | 0.33 | 0.12 |
| I2 k=4 R=2 binding [i2_k4_r2_b] | 4.9 | 44.01 | 91.52 | 99.05 | 0.63 | 98.64 | 0.38 | 0.21 |
| I2 k=8 R=1 [i2_k8_r1] | 4.9 | 44.87 | 92.33 | 99.26 | 0.49 | 99.34 | 0.47 | 0.21 |
| I1 best-of-8 [i1_n16] | 4.91 | 44.65 | 92.73 | 99.27 | 0.57 | 97.8 | 0.34 | 0.23 |
| I1 best-of-16 [i1_n16] | 4.91 | 44.39 | 92.14 | 99.2 | 1.4 | 97.47 | 0.37 | 0.16 |

## The search

| run | hemistichs | first candidate passes | passed | checks / hemistich | rounds / hemistich | sadr passed | ʿajuz passed | poems with a forced hemistich | early end of text / poem | poems with an empty line |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| I2 k=1 R=4 [i2_k1_r4] | 4,264 | 57.08 | 86.19 | 1.85 | 1.855 | 94.37 | 78.0 | 57.0 | 0.005 | 0 |
| I2 k=1 R=8 [i2_k1_r8] | 4,264 | 58.89 | 94.04 | 2.17 | 2.174 | 98.78 | 89.31 | 34.25 | 0.005 | 0 |
| I2 k=4 R=1 [i2_k4_r1] | 4,264 | 56.64 | 85.27 | 1.88 | 1.0 | 94.23 | 76.31 | 59.25 | 0.005 | 0 |
| I2 k=4 R=2 [i2_k4_r2] | 4,264 | 58.79 | 93.79 | 2.14 | 1.116 | 98.69 | 88.88 | 33.75 | 0.003 | 0 |
| I2 k=4 R=2 binding [i2_k4_r2_b] | 4,264 | 51.57 | 89.68 | 2.55 | 1.172 | 97.0 | 82.36 | 50.25 | 0.0 | 0 |
| I2 k=8 R=1 [i2_k8_r1] | 4,264 | 57.43 | 93.93 | 2.15 | 1.0 | 98.41 | 89.45 | 33.75 | 0.005 | 0 |

## Integrity checks (all must be 0)

| condition | missing binding records | missing scored rows | search poems not of the requested length | search poems with no forced hemistich that are not strictly valid | line counts differing from the records |
|---|---:|---:|---:|---:|---:|
| I0 single sample [i1_n16] | 0 | 0 | – | – | 0 |
| I1 best-of-2 [i1_n16] | 0 | 0 | – | – | 0 |
| I1 best-of-4 [i1_n16] | 0 | 0 | – | – | 0 |
| I1 best-of-8 [i1_n16] | 0 | 0 | – | – | 0 |
| I1 best-of-16 [i1_n16] | 0 | 0 | – | – | 0 |
| I2 k=1 R=4 [i2_k1_r4] | 0 | 0 | 0 | 0 | 0 |
| I2 k=1 R=8 [i2_k1_r8] | 0 | 0 | 0 | 0 | 0 |
| I2 k=4 R=1 [i2_k4_r1] | 0 | 0 | 0 | 0 | 0 |
| I2 k=4 R=2 [i2_k4_r2] | 0 | 0 | 0 | 0 | 0 |
| I2 k=4 R=2 binding [i2_k4_r2_b] | 0 | 0 | 0 | 0 | 0 |
| I2 k=8 R=1 [i2_k8_r1] | 0 | 0 | 0 | 0 | 0 |


  ☁ Drive: everything this step wrote is on Drive (0 upload(s), the last 0)


In [5]:
if RUN_M4:
  pv.m4_test_from_choice(pvc)   # GPU: the pre-registered test runs

  Drive: results are uploaded through the Drive API and verified (MD5)
  Drive → VM: m4: 70 file(s), 269.0 MB in 01:02
test runs: search k=1 R=8 and best-of-4
  Generating train split         [░░░░░░░░░░░░░░░░░░░░]   0%  0/109070  elapsed 00:00
  Generating train split         [████░░░░░░░░░░░░░░░░]  19%  21000/109070  elapsed 00:00
  Generating train split         [██████░░░░░░░░░░░░░░]  29%  32000/109070  elapsed 00:00  (100,475.7 it/s)  ETA 00:00
  Generating train split         [████████░░░░░░░░░░░░]  39%  43000/109070  elapsed 00:00  (104,837.1 it/s)  ETA 00:00
  Generating train split         [██████████░░░░░░░░░░]  50%  54000/109070  elapsed 00:00  (100,475.7 it/s)  ETA 00:00
  Generating train split         [████████████░░░░░░░░]  60%  65535/109070  elapsed 00:00  (104,837.1 it/s)  ETA 00:00
  Generating train split         [██████████████░░░░░░]  71%  77535/109070  elapsed 00:00  (102,852.6 it/s)  ETA 00:00
  Generating train split         [████████████████░░░░]  82%  89535/10

In [6]:
if RUN_M4:
  pv.m4_score(pvc, "test")
  pv.m4_report(pvc, "test")
  pv.m4_export(pvc, "test")    # the same for the test runs: m4/test/export/*.csv, all_runs.csv for analysis

  i1_n4/letters: 13,924 poems, 4,840 already scored, 9,084 to score
  … M4 verifier i1_n4: running 00:21 | silent for 00:20 | GPU 0% busy, 0.0/22.5 GB | CPU 7% | idle
  … M4 verifier i1_n4: running 00:41 | silent for 00:40 | GPU 0% busy, 0.0/22.5 GB | CPU 0% | idle
  … M4 verifier i1_n4: running 01:01 | silent for 01:00 | GPU 0% busy, 0.0/22.5 GB | CPU 0% | idle
  … M4 verifier i1_n4: running 01:21 | silent for 01:20 | GPU 0% busy, 0.0/22.5 GB | CPU 0% | idle
  … M4 verifier i1_n4: running 01:41 | silent for 01:40 | GPU 0% busy, 0.0/22.5 GB | CPU 0% | idle
  … M4 verifier i1_n4: running 02:01 | silent for 02:00 | GPU 0% busy, 0.0/22.5 GB | CPU 0% | idle
  … M4 verifier i1_n4: running 02:21 | silent for 02:20 | GPU 0% busy, 0.0/22.5 GB | CPU 0% | idle
  i1_n4/letters: 5,040/13,924 poems, 1.3/s, about 111 min left
  … M4 verifier i1_n4: running 02:51 | silent for 00:20 | GPU 0% busy, 0.0/22.5 GB | CPU 0% | idle
  … M4 verifier i1_n4: running 03:11 | silent for 00:40 | GPU 0% busy, 0.0/22

KeyboardInterrupt: 

In [ ]:
if RUN_M4:
  pv.m4_check_sampling(pvc)     # optional, GPU: the sample mode against the baseline's E1 poems

## 8. M5 - self-training on verifier-approved poems (idea 2)

The released model is never told which rhyme to keep, and rhyme is where it fails most (38% of its test poems pass the
rhyme check, 93% of the human ones). M5 trains new adapters with the baseline's recipe (QLoRA on Yehia-7B-preview,
`sft/train_sft.py` unchanged) on new data:

* **2a**: every prompt names the rhyme letter of its poem (`حرف الروي: ...`, the verifier's rawī), so the model learns to
  follow a requested rhyme; at test time it is asked for the rawī of the human poem of the prompt.
* **2b**: 2a, trained only on the poems the verifier accepts (meter with no margin, and rhyme).
* **2c**: rejection-sampling rounds: the 2a/2b model writes poems for train prompts with M4's hemistich search, a quality
  gate keeps the strictly valid ones that copy nothing from the corpus, and one epoch of training on them (mixed with as
  many original poems) continues the adapter. Up to two rounds.

Each variant is judged by **one ordinary sample per test prompt** (no search, no selection) against the baseline's E3
(the same recipe without the rhyme field) and E1 (the released adapter), and optionally the authors' best GRPO adapter.

**Where things go.** Checkpoints: `MyDrive/shaer-repro-yehia-prosody/sft_outputs/train/prosody_m5_<variant>/`, beside the
baseline's folder, which is never written. A resume checkpoint is written about every 10 minutes. Data, labels and results:
`MyDrive/shaer-prosody/m5/`. Every step resumes.

**Cost.** One training run of 2a or 2b costs about as much as E3 (two epochs of the train split: roughly 12-20 h on an
A100). A 2c round: the search on 4,000 train prompts (1-2 h) and one short epoch (2-3 h). Each test generation: about
1.5 h. Run 2a first; go on to 2b and 2c only if 2a's report shows a gain. The validation is in
`prosody-verifier/validation/M5_report.md`.

In [ ]:
if RUN_M5:
  pv.m5_relabel(pvc)                 # CPU: the verifier's labels of the corpus (rawī, meter, rhyme); resumable

In [ ]:
if RUN_M5:
  pv.m5_build(pvc, "2a")             # CPU: the training set with the rhyme letter in every prompt

In [ ]:
if RUN_M5:
  pv.m5_train(pvc, "2a")             # GPU: training with live progress; re-run the cell to resume

In [ ]:
if RUN_M5:
  pv.m5_generate(pvc, "2a")          # GPU: one sample per test prompt with the best 2a checkpoint
  pv.m5_import_baselines(pvc)        # the baseline's E1 / E3 test poems (read only), for the comparison

In [ ]:
if RUN_M5:
  pv.m5_score(pvc)                   # CPU
  pv.m5_report(pvc)
  pv.m5_export(pvc)            # every M5 poem with its prompt and scores: m5/test/export/*.csv

**Only if 2a gains** (strict validity and rhyme above E3, the quality section no worse): 2b, then the 2c rounds.
`pv.m5_generate(pvc, "grpo_v2")` adds the authors' best GRPO adapter to the comparison (GPU, about 1.5 h).

In [ ]:
if RUN_M5:
  pv.m5_build(pvc, "2b")
  pv.m5_train(pvc, "2b")
  pv.m5_generate(pvc, "2b")

In [ ]:
if RUN_M5:
  pv.m5_selftrain(pvc, round_no=1, base="2a")    # or base="2b" if it did better; GPU
  pv.m5_generate(pvc, "2c_r1")

In [ ]:
if RUN_M5:
  pv.m5_score(pvc)
  pv.m5_report(pvc)
  pv.m5_export(pvc)

## 9. M6 - quality of the final systems

Optimising against the verifier can make poems valid and worse. M6 measures what the verifier cannot, on the final
systems (the M4 test conditions, the M5 models and the human poems of the same prompts):

* the paper's **LLM judge** (description adherence, meaning, fluency, coherence, poeticness; 1-5), on the same prompts for
  every system. It is a **paid API** (OpenRouter, one poem = five calls; the Colab secret `OPENROUTER_API_KEY`): start
  with `dry_run=True`, then a few rows;
* **copying and diversity** (lines copied from the training poems, repeated lines, distinct n-grams, distinct rhyme words);
* a **blinded human study**: a rating sheet (CSV for Google Sheets) with the systems' poems shuffled under item codes, and
  its key kept apart; filled copies go to `MyDrive/shaer-prosody/m6/human/filled/`.

Summarise M4's test runs (`pv.m4_report(pvc, "test")`) and M5's first. A CPU runtime is enough. The validation is in
`prosody-verifier/validation/M6_report.md`.

In [ ]:
if RUN_M6:
  pv.m6_collect(pvc)                  # one poem per test prompt for every final system, and the human poems

In [ ]:
if RUN_M6:
  pv.m6_judge(pvc, dry_run=True)      # the plan and one exact prompt; no API call, no cost

In [ ]:
if RUN_M6:
  pv.m6_judge(pvc, limit_rows=300, max_cost_usd=10)   # paid: the same 300 prompts for every system; resumable

In [ ]:
if RUN_M6:
  SHEET = pv.m6_human_sheet(pvc, names=("human", "e3_retrained", "m5_2a"), prompts=40)   # pick the systems to rate
  # give each rater a copy; put the filled CSVs in MyDrive/shaer-prosody/m6/human/filled/, then:
  # pv.m6_human_score(pvc)

In [ ]:
if RUN_M6:
  pv.m6_report(pvc)

In [ ]:
if RELEASE_RUNTIME:
  pv.release_runtime(pvc)   # uploads what is missing, verifies it, then ends the session; keeps it if anything is not on Drive

## Next milestones

M2 (the expert gold set) is added when a prosody expert is available. Pull the latest `dev` (section 2) before running them.
If you save this notebook from Colab, save it under a different name (or reload it from GitHub first), so a stale copy does
not overwrite newer cells.